# Multimodal Cancer Modeling in the Age of Foundation Model Embeddings
### Steven Song\*, Morgan Borjigin-Wang\*, Irene R. Madejski, Robert L. Grossman

\* Equal contribution

Read our paper here: https://proceedings.mlr.press/v297/song26a.html

***

The Cancer Genome Atlas (TCGA) has enabled novel discoveries and served as a large-scale reference dataset in cancer through its harmonized genomics, clinical, and imaging data. Numerous prior studies have developed bespoke deep learning models over TCGA for tasks such as cancer survival prediction. A modern paradigm in biomedical deep learning is the development of foundation models (FMs) to derive feature embeddings agnostic to a specific modeling task. Biomedical text especially has seen growing development of FMs. While TCGA contains free-text data as pathology reports, these have been historically underutilized.

* **We investigate the ability to train classical machine learning models over multimodal, zero-shot FM embeddings of cancer data.**
* We demonstrate the ease and additive effect of multimodal fusion, outperforming unimodal models.
* Overall, we propose a simple, modernized approach to multimodal cancer modeling using FM embeddings.

### Overview

<img src="https://raw.githubusercontent.com/StevenSong/multimodal-cancer-modeling/refs/heads/main/overview.png" alt="conceptual overview figure" width="50%"/>

Conceptually, the proposed framework does late fusion of unimodal models trained over their respective embeddings. Specifically, we use:
* BulkRNABert ([Gélard et al. 2025)](https://proceedings.mlr.press/v259/gelard25a.html)) for RNA-seq data
* UNI2-h ([Chen et al. 2024](https://www.nature.com/articles/s41591-024-02857-3)) for histology data
* BioMistral ([Labrak et al. 2024](https://aclanthology.org/2024.findings-acl.348/)) for pathology report data (summarized by Llama-3.1-8B-Instruct ([Grattafiori et al. 2024](https://arxiv.org/abs/2407.21783)))

#### Use the GDC API to get TCGA case metadata

In [1]:
import requests
import numpy as np
import pandas as pd

In [2]:
# case metadata
tcga_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}]}
response = requests.post(
    "https://api.gdc.cancer.gov/cases",
    json={
        "filters": tcga_filter,
        "fields": ",".join(["project.project_id", "submitter_id", "diagnoses.age_at_diagnosis", "diagnoses.diagnosis_is_primary_disease", "demographic.sex_at_birth", "demographic.race", "demographic.ethnicity"]),
        "format": "JSON",
        "size": str(100_000),
    },
)

hits = []
for hit in response.json()["data"]["hits"]:
    proj = hit.pop("project", {})
    demo = hit.pop("demographic", {})
    dxs = hit.pop("diagnoses", [])
    for dx in dxs:
        if dx.get("diagnosis_is_primary_disease", False) and dx["age_at_diagnosis"] is not None and not np.isnan(dx["age_at_diagnosis"]):
            hit["age_at_diagnosis"] = dx["age_at_diagnosis"]
            break
    if "age_at_diagnosis" in hit:
        hits.append(hit | proj | demo)
metadata = pd.DataFrame(hits).drop(columns=["id"])
assert metadata["age_at_diagnosis"].notna().all()
metadata["age_in_years"] = metadata["age_at_diagnosis"] / 365.24 # convert age to years
metadata["age_binned"] = pd.cut(metadata["age_in_years"], bins=[0, 20, 40, 60, 80, 100]) # convert age to 20-year bins
metadata

,submitter_id,age_at_diagnosis,project_id,race,ethnicity,sex_at_birth,age_in_years,age_binned
0,TCGA-44-3918,22236,TCGA-LUAD,white,not hispanic or latino,female,60.880517,"(60, 80]"
1,TCGA-44-6146,23378,TCGA-LUAD,white,not hispanic or latino,male,64.007228,"(60, 80]"
2,TCGA-62-8394,23758,TCGA-LUAD,white,not hispanic or latino,female,65.047640,"(60, 80]"
3,TCGA-55-8204,31818,TCGA-LUAD,white,not hispanic or latino,female,87.115321,"(80, 100]"
4,TCGA-MN-A4N1,21939,TCGA-LUAD,black or african american,not hispanic or latino,male,60.067353,"(60, 80]"
...,...,...,...,...,...,...,...,...
11066,TCGA-BP-4803,28934,TCGA-KIRC,white,not reported,male,79.219144,"(60, 80]"
11067,TCGA-CZ-5461,19030,TCGA-KIRC,white,not hispanic or latino,male,52.102727,"(40, 60]"
11068,TCGA-CJ-4907,21208,TCGA-KIRC,white,not hispanic or latino,male,58.065929,"(40, 60]"
11069,TCGA-BP-4969,23216,TCGA-KIRC,white,not hispanic or latino,female,63.563684,"(60, 80]"


In [3]:
# survival data
response = requests.post(
    "https://api.gdc.cancer.gov/analysis/survival",
    json={"filters": tcga_filter},
)
rows = response.json()["results"][0]["donors"]
survival = pd.DataFrame(rows).drop(columns=["id", "project_id"])
survival

,time,censored,survivalEstimate,submitter_id
0,1.0,True,0.999278,TCGA-C8-A275
1,1.0,False,0.999278,TCGA-AA-3492
2,1.0,True,0.999278,TCGA-AC-A7VC
3,1.0,True,0.999278,TCGA-FV-A495
4,1.0,False,0.999278,TCGA-CG-4306
...,...,...,...,...
11075,9634.0,True,0.164279,TCGA-WB-A80P
11076,10346.0,False,0.123209,TCGA-EE-A2GD
11077,10870.0,False,0.082140,TCGA-FS-A1ZC
11078,11217.0,True,0.082140,TCGA-LH-A9QB


#### Load embeddings from the Gen3 Embedding Service

In [4]:
from gen3.auth import Gen3Auth
from tqdm import tqdm
from tqdm.contrib.concurrent import thread_map

auth = Gen3Auth()

EMBEDDING_API = "https://genomicaicommons.org/ai/vectorstore/collections"
PAGE_SIZE = 100
COLLECTIONS = ["expr", "hist", "summ"]

In [5]:
r = requests.get(f"{EMBEDDING_API}", params={"counts": True}, auth=auth)
r.raise_for_status()
collection_count = {
    x["collection_name"]: x["available_embeddings_count"]
    for x in r.json()["collections"] if x["collection_name"] in COLLECTIONS
}
collection_count

{'expr': 8203, 'hist': 9751, 'summ': 9523}

In [6]:
EMB_T = list[float]
META_T = dict[str, str]

def paged_query_factory(_collection: str):
    def _paged_query(page: int) -> list[tuple[EMB_T, META_T]]:
        r = requests.get(f"{EMBEDDING_API}/{_collection}/embeddings", params={"page": page, "page_size": PAGE_SIZE}, auth=auth)
        r.raise_for_status()
        return [(x["vector"], x["info"]["metadata"]) for x in r.json()["embeddings"]]
    return _paged_query

data = {}
for collection in COLLECTIONS:
    paged_query = paged_query_factory(collection)
    n = collection_count[collection]
    pages = [(i // PAGE_SIZE) + 1 for i in range(0, n, PAGE_SIZE)]
    xs = thread_map(paged_query, pages, max_workers=8, desc=collection)
    data[collection] = {
        "embs": np.asarray([e for batch in xs for e, m in batch]),
        "meta": pd.DataFrame([m for batch in xs for e, m in batch]),
    }

expr:   0%|          | 0/83 [00:00<?, ?it/s]

hist:   0%|          | 0/98 [00:00<?, ?it/s]

summ:   0%|          | 0/96 [00:00<?, ?it/s]

#### Align metadata and embeddings

In [7]:
# case IDs that have all 3 embedding modalities
case_ids = set.intersection(*[set(data[c]["meta"]["case_id"]) for c in COLLECTIONS])

assert metadata["submitter_id"].is_unique and survival["submitter_id"].is_unique
df = metadata.merge(survival, on="submitter_id")
df = df[df["submitter_id"].isin(case_ids)]
df = df.sort_values(["project_id", "submitter_id"])
df = df.set_index("submitter_id")
df

,age_at_diagnosis,project_id,race,ethnicity,sex_at_birth,age_in_years,age_binned,time,censored,survivalEstimate
submitter_id,,,,,,,,,,
TCGA-OR-A5J1,21496,TCGA-ACC,white,not reported,male,58.854452,"(40, 60]",1355.0,False,0.645187
TCGA-OR-A5J2,16090,TCGA-ACC,white,hispanic or latino,female,44.053225,"(40, 60]",1677.0,False,0.591072
TCGA-OR-A5J3,8624,TCGA-ACC,white,hispanic or latino,female,23.611872,"(20, 40]",2091.0,True,0.536551
TCGA-OR-A5J5,11171,TCGA-ACC,white,hispanic or latino,male,30.585369,"(20, 40]",365.0,False,0.881805
TCGA-OR-A5J6,10839,TCGA-ACC,black or african american,hispanic or latino,female,29.676377,"(20, 40]",2703.0,True,0.473618
...,...,...,...,...,...,...,...,...,...,...
TCGA-YZ-A980,27716,TCGA-UVM,white,not hispanic or latino,male,75.884350,"(60, 80]",1862.0,True,0.565376
TCGA-YZ-A982,28938,TCGA-UVM,white,not hispanic or latino,female,79.230095,"(60, 80]",495.0,True,0.833562
TCGA-YZ-A983,18769,TCGA-UVM,white,not hispanic or latino,female,51.388128,"(40, 60]",798.0,True,0.751077


In [8]:
# align ordering of embeddings to case order in metadata
for c in COLLECTIONS:
    meta = data[c]["meta"]
    embs = data[c]["embs"]
    case_embs = []
    for case_id in tqdm(df.index, desc=c):
        idxs = meta[meta["case_id"] == case_id].index
        _case_emb = embs[idxs].mean(axis=0)
        case_embs.append(_case_emb)
    data[c]["case_embs"] = np.asarray(case_embs)

summ: 100%|██████████| 7893/7893 [00:07<00:00, 1044.12it/s]


#### Preprocess data before modeling

In [9]:
from typing import Optional
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

In [10]:
demo_X = OneHotEncoder(drop="if_binary", sparse_output=False, dtype=np.float32).fit_transform(df[["sex_at_birth", "age_binned", "race", "ethnicity"]])
canc_X = OneHotEncoder(drop="if_binary", sparse_output=False, dtype=np.float32).fit_transform(df[["project_id"]])
expr_X = data["expr"]["case_embs"]
hist_X = data["hist"]["case_embs"]
text_X = data["summ"]["case_embs"]
y = np.asarray(
    list(zip(~df["censored"], df["time"])),
    dtype=[("Status", "?"), ("Survival_in_days", "<f8")],
)

In [11]:
# stratify by observed mortality and cancer type
splitter = df["censored"].astype(str) + "_" + df["project_id"]

# split all data modalities into train/test
(
    demo_X_train, demo_X_test,
    canc_X_train, canc_X_test,
    expr_X_train, expr_X_test,
    hist_X_train, hist_X_test,
    text_X_train, text_X_test,
    y_train,      y_test,
) = train_test_split(
    demo_X, canc_X, expr_X, hist_X, text_X, y,
    test_size=0.2,
    random_state=42,
    stratify=splitter,
)

In [12]:
# this one helper will be used to run both unimodal and multimodal experiments
def train_eval_model(
    *,  # enforce kwargs
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray,
    pca_components: Optional[int], standardize: bool,
) -> dict:
    print("Training survival model")

    # z-score input features
    if standardize:
        print("--standardized")
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    # dimensionality reduction
    if pca_components is not None:
        print("--reduced")
        pca = PCA(n_components=pca_components, random_state=42)
        X_train = pca.fit_transform(X_train)
        X_test = pca.transform(X_test)

    # fit survival model
    cox = CoxPHSurvivalAnalysis(alpha=0.1).fit(X_train, y_train)
    print("--trained")

    # generate predictions
    y_train_pred = cox.predict(X_train)
    y_test_pred = cox.predict(X_test)

    # evaluate predictions
    c_index = concordance_index_censored(
        event_indicator=y_test["Status"],
        event_time=y_test["Survival_in_days"],
        estimate=y_test_pred,
    )[0]

    return {
        "c_index": c_index,
        "y_test_pred": y_test_pred,
        "y_train_pred": y_train_pred,
    }

#### Train unimodal models

In [13]:
demo_results = train_eval_model(X_train=demo_X_train, y_train=y_train, X_test=demo_X_test, y_test=y_test, pca_components=None, standardize=False)
canc_results = train_eval_model(X_train=canc_X_train, y_train=y_train, X_test=canc_X_test, y_test=y_test, pca_components=None, standardize=False)
expr_results = train_eval_model(X_train=expr_X_train, y_train=y_train, X_test=expr_X_test, y_test=y_test, pca_components=256, standardize=True)
hist_results = train_eval_model(X_train=hist_X_train, y_train=y_train, X_test=hist_X_test, y_test=y_test, pca_components=256, standardize=True)
text_results = train_eval_model(X_train=text_X_train, y_train=y_train, X_test=text_X_test, y_test=y_test, pca_components=256, standardize=True)

Training survival model
--trained
Training survival model
--trained
Training survival model
--standardized
--reduced
--trained
Training survival model
--standardized
--reduced
--trained
Training survival model
--standardized
--reduced
--trained


#### Train multimodal model
The unimodal models' predicted risk scores are used as input to the multimodal fusion model.

In [14]:
fuse_X_train = np.asarray([
    demo_results["y_train_pred"],
    canc_results["y_train_pred"],
    expr_results["y_train_pred"],
    hist_results["y_train_pred"],
    text_results["y_train_pred"],
]).T

fuse_X_test = np.asarray([
    demo_results["y_test_pred"],
    canc_results["y_test_pred"],
    expr_results["y_test_pred"],
    hist_results["y_test_pred"],
    text_results["y_test_pred"],
]).T

fuse_results = train_eval_model(X_train=fuse_X_train, y_train=y_train, X_test=fuse_X_test, y_test=y_test, pca_components=None, standardize=True)

Training survival model
--standardized
--trained


#### Evaluation
Our multimodal fusion approach substantially beats all unimodal results!

In [15]:
results = pd.Series({
    "demo": demo_results["c_index"],
    "canc": canc_results["c_index"],
    "expr": expr_results["c_index"],
    "hist": hist_results["c_index"],
    "text": text_results["c_index"],
    "fuse": fuse_results["c_index"],
}, name="C-index")

print("Unimodal Results")
print("------------------")
display(results.loc[["demo", "canc", "expr", "hist", "text"]])

print("Multimodal Results")
print("------------------")
display(results.loc[["fuse"]])

Unimodal Results
------------------


demo    0.612508
canc    0.742682
expr    0.756662
hist    0.769448
text    0.742404
Name: C-index, dtype: float64

Multimodal Results
------------------


fuse    0.791862
Name: C-index, dtype: float64